# omnivoice: обучение голоса в Colab

Выбери среду выполнения с GPU: *Среда выполнения → Сменить тип → GPU*. Затем запускай ячейки по порядку.

**Если Colab отключился:** заново запусти установку (2), загрузку датасета (3), Google Drive (если использовал) и ячейку «Настройка». Обучение продолжится с последнего чекпойнта `last.ckpt`.

## 1. Проверка видеокарты

In [ ]:
!nvidia-smi

## 2. Установка piper и omnivoice

Занимает несколько минут. Ячейку можно запускать повторно.

In [ ]:
!if [ ! -d piper1-gpl ]; then git clone --depth 1 --branch v1.8.0 https://github.com/OHF-voice/piper1-gpl.git; fi && cd piper1-gpl && pip install onnx onnxscript && pip install -e '.[train]' && ./build_monotonic_align.sh && python3 setup.py build_ext --inplace
!pip install "git+https://github.com/Shura4eburek/OmniChat.git#subdirectory=tools/omnivoice"

## 3. Загрузка датасета

На компьютере выполни `omnivoice train --colab` — получишь файл `<имя>-dataset.zip`. Загрузи его сюда.

In [ ]:
import glob, os, zipfile
from google.colab import files
uploaded = files.upload()
zips = sorted(n for n in uploaded if n.endswith("-dataset.zip")) or sorted(n for n in uploaded if n.endswith(".zip"))
assert zips, "Не загружен файл *-dataset.zip"
os.makedirs("/content/project", exist_ok=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall("/content/project")
print("Распаковано:", zips[0])

## 4. Google Drive (необязательно)

Чекпойнты будут сохраняться на Drive и переживут отключение Colab. Существующие локальные чекпойнты переносятся на Drive, ничего не удаляется.

In [ ]:
import os, shutil
from pathlib import Path
from google.colab import drive
from omnivoice.project import Project
drive.mount("/content/drive")
p = Project.load(Path("/content/project"))
drive_dir = Path("/content/drive/MyDrive/omnivoice-train") / p.name
drive_dir.mkdir(parents=True, exist_ok=True)
local = Path("/content/project/train/lightning_logs")
local.parent.mkdir(parents=True, exist_ok=True)
if local.is_symlink():
    print("Уже подключено к Drive")
else:
    if local.is_dir():
        for child in list(local.iterdir()):
            if (drive_dir / child.name).exists():
                print("На Drive уже есть, пропускаю:", child.name)
            else:
                shutil.move(str(child), str(drive_dir / child.name))
        try:
            local.rmdir()
        except OSError:
            print("Локальная папка не пуста, оставляю как есть:", local)
    if not local.exists():
        local.symlink_to(drive_dir, target_is_directory=True)
        print("Чекпойнты сохраняются в", drive_dir)

## 5. Настройка

Запускай эту ячейку после каждого переподключения. Она включает совместимость piper с новым PyTorch (загрузка чекпойнтов и экспорт в ONNX).


In [ ]:
import os
from pathlib import Path
from omnivoice import piper_compat
piper_compat.install_shim()
os.environ["OMNIVOICE_CACHE"] = "/content/cache"
from omnivoice import checkpoints, train
from omnivoice.project import Project
p = Project.load(Path("/content/project"))
print("Проект:", p.name)

## 6. Базовая модель

Скачивается чекпойнт, с которого начнётся дообучение.

In [ ]:
os.environ["OMNIVOICE_CACHE"] = "/content/cache"
from omnivoice import piper_compat
base_ckpt = piper_compat.clean_base_local(checkpoints.ensure(p.base_checkpoint))
print(base_ckpt)

## 7. Обучение

Если есть `last.ckpt` (в том числе на Drive), обучение продолжится с него.

In [ ]:
#@markdown **Сколько эпох дообучать поверх базовой модели**
epochs = 1000  #@param {type:"integer"}
batch = 16  #@param {type:"integer"}
import os, subprocess
from pathlib import Path
from omnivoice import checkpoints, piper_compat, train
piper_compat.install_shim()
from omnivoice.project import Project
os.environ["OMNIVOICE_CACHE"] = "/content/cache"
p = Project.load(Path("/content/project"))
last = train.last_checkpoint(p)
if last is not None:
    ckpt = last
    print("Продолжаю с last.ckpt:", last)
else:
    ckpt = piper_compat.clean_base_local(checkpoints.ensure(p.base_checkpoint))
max_epochs = train.target_epochs(p, epochs)
cmd = train.fit_args(p, str(ckpt), batch, max_epochs, "/content/project")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 8. Экспорт и упаковка

In [ ]:
import os, subprocess
from pathlib import Path
from omnivoice import piper_compat, train
piper_compat.install_shim()
from omnivoice.project import Project
p = Project.load(Path("/content/project"))
last = train.last_checkpoint(p)
if last is None:
    raise SystemExit("Нет чекпойнтов — сначала запусти обучение.")
os.makedirs("/content/project/export", exist_ok=True)
r = subprocess.run(["python3", "-m", "piper.train.export_onnx", "--checkpoint", str(last),
                    "--output-file", "/content/project/export/model.onnx"], check=False)
if r.returncode != 0:
    raise SystemExit("Экспорт в ONNX не удался — см. вывод выше.")
r = subprocess.run(["omnivoice", "pack", "--project", "/content/project"], capture_output=True, text=True, check=False)
print(r.stdout, r.stderr)
if r.returncode != 0 or "Голос собран:" not in r.stdout:
    raise SystemExit("Упаковка не удалась — см. сообщения выше.")
voice_dir = Path(r.stdout.split("Голос собран:", 1)[1].strip().splitlines()[0].strip())
v = subprocess.run(["omnivoice", "verify", str(voice_dir)], check=False)
if v.returncode != 0:
    print("Проверка голоса не прошла — см. вывод выше.")

## 9. Скачать голос

Распакуй архив — получится папка с голосом. Положи её в папку `models` мода OmniChat или выполни `omnivoice install`.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(f"/content/{voice_dir.name}", "zip", root_dir=str(voice_dir.parent), base_dir=voice_dir.name)
files.download(archive)